### Raw to Landing — HR Domain
**Author:** Virendra Tambavekar 
**Task:** Ingest HR domain data from raw to landing layer 
**Domain:** HR (Broker) 
**Pipeline Stage:** Raw → Landing

In [0]:
%run ../../02_common_utils/raw_to_landing

In [0]:
%run ../../02_common_utils/operations

In [0]:
recon_df=load_domain("hr_broker")

In [0]:
# Configuration
CATALOG = "charles_schwab_retailbrokerage_dev_team_lemma"
spark.sql(f"USE CATALOG {CATALOG}")

# Generate run_id using operations utility (raw -> landing has no upstream _run_id)
carried_run_id = generate_run_id()
source_count = recon_df.select("source_count").first()[0]
target_count = recon_df.select("target_count").first()[0]
print(f"Generated run_id: {carried_run_id}")

# __ start_pipeline_run -- imported from operations
start_pipeline_run(spark=spark, run_id=carried_run_id, batch="ALL")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="RUNNING")

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="raw_to_landing_hr", message="Pipeline started: Raw to Landing ingestion for HR domain")

# __ log_pipeline_recon -- imported from operations
log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="ALL",
    domain="HR",
    table_name="hr_broker",
    source_layer="raw",
    target_layer="landing",
    source_count=source_count,
    target_count=target_count
)

# __ log_audit_event -- imported from operations
log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="ALL",
    layer="landing",
    table_name="hr_broker",
    operation="OVERWRITE",
    rows_affected=target_count
)

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="raw_to_landing_hr", message=f"Pipeline completed: {target_count} rows written to landing.hr_broker")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="COMPLETED")

# __ end_pipeline_run -- imported from operations
end_pipeline_run(spark=spark, run_id=carried_run_id, status="SUCCESS")

print(f"Operations logging complete for run_id: {carried_run_id}")

In [0]:
from pyspark.sql import SparkSession
def log_dq_result(spark: SparkSession, run_id: str, table_name: str, rule_name: str, failed_rows: int, total_rows: int):

    """
    Logs the outcome of a Data Quality (DQ) check.
    """
    status = 'PASS' if failed_rows == 0 else 'FAIL'
    
    spark.sql(f"""
        INSERT INTO operations.dq_results 
        (run_id, table_name, rule_name, failed_rows, total_rows, dq_status)
        VALUES ('{run_id}', '{table_name}', '{rule_name}', {failed_rows}, {total_rows}, '{status}')
    """)